# Day 2 - Topic 5: @property, Static Methods, and Class Methods

> Lead-Level Data Science Interview Prep Series

## 1. Introduction

- **@property** turns a method into something accessed like an attribute (no parentheses), letting you add logic behind attribute access
- **@staticmethod** marks a method that needs NEITHER the instance (self) NOR the class - just a related utility living inside the class
- **@classmethod** marks a method that receives the CLASS (cls) instead of the instance - commonly used for alternative constructors
- Why needed?
  - @property: computed/validated attributes without changing the calling code (`obj.area` instead of `obj.area()`)
  - Alternative constructors: `DataFrame.from_dict(...)`, `datetime.fromtimestamp(...)` - both are classmethods
- Where used?
  - `df.shape`, `df.columns` in Pandas are properties (that is why there are no parentheses)
  - `pd.DataFrame.from_records(...)` is a classmethod
  - Standard interview question: difference between staticmethod and classmethod

## 2. Real-Life Analogy

- **@property** = a car's fuel gauge: it LOOKS like a simple dial (attribute), but behind the glass it actively computes the level from the tank sensor every time you glance at it. You never "call" the gauge - you just read it
- **Setter with validation** = the fuel tank's inlet: it looks like a simple opening, but it rejects the wrong nozzle (validation happens on assignment)
- **@staticmethod** = a calculator lying in the office drawer: related to the office's work, stored there for convenience, but it doesn't know or care WHICH office it is in
- **@classmethod** = the company's official registration desk: it acts on behalf of the whole company (class), not one employee (instance) - e.g. "create a new employee record from this ID card" (alternative constructor)

## 3. Explanation

- **@property**
  - Decorate a method with `@property` -> read it as `obj.name` (no parentheses)
  - Add `@name.setter` to control what happens on assignment (`obj.name = value`) - the place for validation
  - Without a setter, the property is effectively read-only
- **@staticmethod**
  - No automatic first argument at all - behaves like a plain function namespaced inside the class
  - Callable from the class or an instance
- **@classmethod**
  - First argument is `cls` (the class itself), passed automatically
  - Main use: alternative constructors - different ways to build an instance
  - Respects inheritance: called on a subclass, `cls` IS the subclass

> **Trick to remember:** self = this object, cls = this class, static = neither. Property = method wearing an attribute costume.

## 4. Syntax

```python
class Circle:
    def __init__(self, radius):
        self._radius = radius          # underscore prefix = "internal, do not touch directly"

    @property
    def radius(self):                  # getter - read as c.radius
        return self._radius

    @radius.setter
    def radius(self, value):           # runs on c.radius = value
        if value <= 0:
            raise ValueError("Radius must be positive")
        self._radius = value

    @property
    def area(self):                    # computed, read-only property
        return 3.14159 * self._radius ** 2

    @staticmethod
    def describe():                    # no self, no cls
        return "A circle is a round shape"

    @classmethod
    def from_diameter(cls, d):         # alternative constructor
        return cls(d / 2)              # cls(...) = create an instance
```

- `_radius` - single leading underscore is a convention meaning "internal/private-ish"
- `cls(d / 2)` - calling the class itself to build a new object

In [ ]:
class Circle:
    def __init__(self, radius):
        self._radius = radius

    @property
    def radius(self):
        return self._radius

    @radius.setter
    def radius(self, value):
        if value <= 0:
            raise ValueError("Radius must be positive")
        self._radius = value

    @property
    def area(self):
        return 3.14159 * self._radius ** 2

    @classmethod
    def from_diameter(cls, d):
        return cls(d / 2)

c = Circle(5)
print(c.radius)               # property - no parentheses
print(round(c.area, 2))       # computed on the fly
c.radius = 10                 # setter runs (validates)
print(round(c.area, 2))       # area automatically reflects new radius

c2 = Circle.from_diameter(20) # alternative constructor
print(c2.radius)


## 5. Examples

### Basic Example

In [ ]:
# Basic: read-only computed property
class Employee:
    def __init__(self, first, last):
        self.first = first
        self.last = last

    @property
    def full_name(self):
        return f"{self.first} {self.last}"

e = Employee("Asha", "Patel")
print(e.full_name)        # looks like an attribute, computed from parts

e.first = "Neha"
print(e.full_name)        # always up to date - no stale stored copy


### Intermediate Example

In [ ]:
# Intermediate: setter validation + all three method types together
class Temperature:
    def __init__(self, celsius=0):
        self.celsius = celsius            # goes through the setter below!

    @property
    def celsius(self):
        return self._celsius

    @celsius.setter
    def celsius(self, value):
        if value < -273.15:
            raise ValueError("Below absolute zero is impossible")
        self._celsius = value

    @property
    def fahrenheit(self):                 # computed view of the same data
        return self._celsius * 9 / 5 + 32

    @staticmethod
    def is_freezing(celsius_value):       # pure utility - no object needed
        return celsius_value <= 0

    @classmethod
    def from_fahrenheit(cls, f):          # alternative constructor
        return cls((f - 32) * 5 / 9)

t = Temperature.from_fahrenheit(98.6)
print(round(t.celsius, 1))                # 37.0
print(Temperature.is_freezing(-5))        # True - called on the class directly

try:
    t.celsius = -300                      # setter blocks invalid state
except ValueError as err:
    print("Blocked:", err)


- Note the subtle line `self.celsius = celsius` in __init__ - it routes through the SETTER, so even construction is validated (one validation point, everywhere)
- fahrenheit is a computed VIEW - one source of truth (_celsius), no duplicate state to keep in sync
- is_freezing needs no instance data -> staticmethod; from_fahrenheit builds instances -> classmethod

### Real-World Example

In [ ]:
# Real-world: a Dataset class mirroring how Pandas/sklearn actually use these tools
class Dataset:
    def __init__(self, records):
        self._records = records           # list of dicts

    @property
    def shape(self):                      # mirrors df.shape - property, not method
        n_rows = len(self._records)
        n_cols = len(self._records[0]) if self._records else 0
        return (n_rows, n_cols)

    @property
    def columns(self):                    # mirrors df.columns
        return list(self._records[0].keys()) if self._records else []

    @staticmethod
    def is_valid_record(record):          # validation utility
        return isinstance(record, dict) and len(record) > 0

    @classmethod
    def from_csv_text(cls, csv_text):     # mirrors pd.read_csv-style alternative construction
        lines = csv_text.strip().split("\n")
        headers = lines[0].split(",")
        records = [dict(zip(headers, line.split(","))) for line in lines[1:]]
        return cls(records)

csv_data = """emp_id,name,dept
101,Asha,Sales
102,Vikram,Engineering
103,Neha,Sales"""

ds = Dataset.from_csv_text(csv_data)
print(ds.shape)        # (3, 3) - no parentheses, exactly like df.shape
print(ds.columns)
print(Dataset.is_valid_record({"a": 1}))


- This explains a daily Pandas detail: `df.shape` has no parentheses because it IS a property, while `df.head()` has them because it is a regular method
- `from_csv_text` follows the same pattern as `pd.DataFrame.from_dict` / `datetime.fromtimestamp` - classmethod constructors named `from_...`
- `dict(zip(headers, values))` is a classic idiom to pair two lists into a dict

## 6. Internal Working

- `@property` creates a **descriptor** object stored on the CLASS - when Python sees `obj.area`, the descriptor protocol intercepts the lookup and calls the getter function behind the scenes
- The same mechanism routes `obj.area = x` to the setter (or raises AttributeError if no setter exists - that is what makes it read-only)
- staticmethod/classmethod are also descriptors: they control what gets passed as the first argument (nothing vs the class)
- On a subclass, a classmethod's `cls` is the SUBCLASS - so inherited alternative constructors build subclass instances correctly

> **Trick to remember:** All three decorators are descriptors - gatekeepers that intercept attribute access on the class. Deep detail is optional; the word "descriptor" in an interview signals depth.

In [ ]:
class Base:
    @classmethod
    def create(cls):
        return cls()          # cls is whatever class you called it on

class Special(Base):
    pass

print(type(Base.create()).__name__)      # Base
print(type(Special.create()).__name__)   # Special - cls respected inheritance


## 7. Time and Space Complexity

- Property access: O(1) dispatch + the getter body's own cost - BUT a property that loops over data (like our shape) re-runs that cost on EVERY access
- staticmethod/classmethod call overhead: O(1), same as normal calls
- Design implication: if a property is expensive to compute and read often, cache the result (e.g. functools.cached_property) - trade space for time

## 8. Common Mistakes

- Storing the property's backing value with the SAME name as the property (self.celsius inside the celsius setter) -> infinite recursion; the backing field must differ (self._celsius)
- Calling a property with parentheses: `c.area()` -> TypeError (float is not callable)
- Forgetting the setter and being surprised that assignment raises AttributeError (that is the read-only behavior)
- Using @staticmethod when the method actually needs class access (creating instances) - should be @classmethod
- Making every attribute a property "just in case" - plain attributes are fine until you need logic; Java-style getters/setters everywhere is un-Pythonic

In [ ]:
# The infinite recursion trap (shown conceptually, not executed fully)
class Bad:
    @property
    def value(self):
        return self.value      # calls the property again -> RecursionError!

# Correct: different backing name
class Good:
    def __init__(self, v):
        self._value = v
    @property
    def value(self):
        return self._value     # reads the underscore-backed field

g = Good(42)
print(g.value)


## 9. Best Practices

- Start with plain attributes; upgrade to @property only when you need validation/computation - callers never notice the change (that is the beauty)
- Back every property with an underscore-prefixed attribute (self._x behind property x)
- Route __init__ assignments through the setter (self.x = x, not self._x = x) so validation applies at construction too
- Name classmethod constructors from_something (from_csv, from_dict, from_config) - instantly recognizable convention
- Choose deliberately: needs instance data -> normal method; needs the class -> classmethod; needs neither -> staticmethod (or move it outside the class entirely)

## 10. Interview Questions

**Beginner**
- Q: What does @property do?
  A: It lets a method be accessed like an attribute (no parentheses), enabling computed or validated values while keeping simple attribute-style syntax for callers.
- Q: Why does df.shape have no parentheses but df.head() does?
  A: shape is defined as a property (attribute-style access); head is a regular method that must be called.

**Intermediate**
- Q: What is the difference between @staticmethod and @classmethod?
  A: A classmethod automatically receives the class as its first argument (cls) and can access class state or create instances; a staticmethod receives nothing automatically - it is a plain function namespaced in the class.
- Q: How do you make a property read-only?
  A: Define only the getter with @property and no setter - assignment then raises AttributeError.

**Advanced**
- Q: Why is @classmethod the right tool for alternative constructors?
  A: Because cls refers to the actual class it is called on - including subclasses - so inherited constructors like from_csv automatically build instances of the subclass, which a hardcoded class name would break.
- Q: What happens mechanically when you access a property?
  A: The property object on the class is a descriptor; attribute lookup finds it and the descriptor protocol invokes its getter (or setter on assignment), instead of returning the stored value directly.

> **Rapid-fire interview line:** self for instance methods, cls for classmethods, nothing for staticmethods; property = computed attribute with optional validation.

## 11. Practice Problems

**Easy**
1. Create a Square class where side is a plain attribute and area is a read-only property.
2. Add a @staticmethod is_valid_side(value) that returns True only for positive numbers.

**Medium**
3. Add a setter to side that raises ValueError for non-positive values, and route the constructor through it. Show that Square(-3) now fails.
4. Add a classmethod from_area(cls, area) that builds a Square from a target area (side = square root of area).

**Hard**
5. Build a BankAccount where balance is a read-only property, deposits/withdrawals happen only through methods (withdraw blocks overdrafts), and a classmethod from_string parses "Asha:5000" format. Explain in a comment why making balance settable would weaken the design.

## 12. Revision Summary

- @property = method accessed like an attribute; add @name.setter for validated assignment; no setter = read-only
- Back properties with underscore fields (self._x); same-name recursion is the classic bug
- Route __init__ through setters for construction-time validation
- @staticmethod: no self, no cls - a utility parked inside the class
- @classmethod: gets cls; the standard tool for from_... alternative constructors; subclass-aware
- df.shape (property) vs df.head() (method) - now you know exactly why
- Decision rule: instance data -> method; class needed -> classmethod; neither -> staticmethod

> **Next topic (Day 2 finale):** Error Handling + Context Managers + Iterators vs Generators